# True Latent ODE v3 — Colab Training

**Model:** True Latent ODE (Chen et al. 2018) — beta-VAE (fixed beta=0.1) + variable-window augmentation (Decision #25)  
**Data:** OLIVES, 77 train eyes → 6,752 variable-window sequences  
**Bars (Decision #23):** RMSE ≤ 85 μm · CI coverage ≥ 80% · KL > 0.1 nats  
**Solver:** rk4 fixed-step (faster than dopri5)  
**W&B run:** `true_latent_ode_v3_seed42` in project `synapse`

**Before running:**
1. Runtime → Change runtime type → T4 GPU
2. Upload `data/processed/olives_sequences_v2.pkl` when prompted in Cell 3

In [ ]:
# Cell 1 — GPU check
import torch
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU. Runtime > Change runtime type > T4 GPU")

In [ ]:
# Cell 2 — Clone repo and install dependencies
!git clone https://github.com/brittLiban/biocomp.git
%cd biocomp
!pip install -q torchdiffeq wandb openpyxl

In [ ]:
# Cell 3 — Upload OLIVES cache (5.6 MB)
# Source: C:\Users\liban\Documents\biocomp\data\processed\olives_sequences_v2.pkl
import os
from google.colab import files

os.makedirs("data/processed", exist_ok=True)
cache_path = "data/processed/olives_sequences_v2.pkl"

if os.path.exists(cache_path):
    print(f"Cache already present ({os.path.getsize(cache_path)/1e6:.1f} MB) — skipping upload.")
else:
    print("Upload olives_sequences_v2.pkl when the file picker appears...")
    uploaded = files.upload()
    for fname in uploaded:
        os.rename(fname, cache_path)
    print(f"Saved to {cache_path} ({os.path.getsize(cache_path)/1e6:.1f} MB)")

In [ ]:
# Cell 4 — W&B login
import wandb
wandb.login()

In [ ]:
# Cell 5 — Verify data loads correctly
import sys
sys.path.insert(0, "src")
import numpy as np
from data.olives import build_sequences, split_by_eye

seqs = build_sequences()
train_seqs, test_seqs = split_by_eye(seqs, test_frac=0.2, seed=42)
print(f"Train eyes: {len(train_seqs)}  |  Test eyes: {len(test_seqs)}")
print(f"Total sequences loaded: {len(seqs)}")

import scripts.true_latent_ode_v3 as v3
all_cst = np.concatenate([s['cst'] for s in train_seqs.values()])
cst_mean, cst_std = float(all_cst.mean()), float(all_cst.std())
train_ds = v3.VariableWindowDataset(train_seqs, cst_mean, cst_std)
print(f"Augmented training sequences: {len(train_ds)} (from {len(train_seqs)} eyes)")
# Expected: ~6752 sequences from 77 eyes

In [ ]:
# Cell 6 — Run training
# Logs to W&B project=synapse, run=true_latent_ode_v3_seed42
# Decision #23 bars: RMSE <= 85 um | CI coverage >= 0.80 | KL > 0.10 nats
# Solver: rk4 fixed-step (faster than dopri5)
# Checkpoint saved to models/true_latent_ode_v3_seed42.pt
!python scripts/true_latent_ode_v3.py

In [ ]:
# Cell 7 — Download checkpoint
from google.colab import files
import os

ckpt = "models/true_latent_ode_v3_seed42.pt"
if os.path.exists(ckpt):
    files.download(ckpt)
    print("Checkpoint downloaded.")
else:
    print("Checkpoint not found — did training complete?")